# Fraud Detection Feature Engineering with EMR on EKS + RAPIDS

This notebook demonstrates the same fraud detection feature engineering pipeline from the original notebook,
but adapted to run on EMR on EKS with NVIDIA RAPIDS acceleration.

**Key Changes from Original:**
- Kubernetes-aware Spark configuration
- EMR on EKS specific settings
- RAPIDS GPU acceleration enabled
- Container resource limits adjusted
- Same business logic and data processing

In [ ]:
# Load Libraries and initialize session
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import year, month, dayofmonth
from pyspark.sql.functions import broadcast
import os

In [ ]:
# Environment Configuration for EMR on EKS
VIRTUAL_CLUSTER_ID = os.environ.get('VIRTUAL_CLUSTER_ID', 'default-cluster')
EMR_EXECUTION_ROLE_ARN = os.environ.get('EMR_EXECUTION_ROLE_ARN')
KUBERNETES_NAMESPACE = os.environ.get('KUBERNETES_NAMESPACE', 'emr-fraud-detection')
CONTAINER_IMAGE = os.environ.get('SPARK_CONTAINER_IMAGE', 'fraud-detection/spark-rapids:latest')

print(f"Virtual Cluster ID: {VIRTUAL_CLUSTER_ID}")
print(f"Kubernetes Namespace: {KUBERNETES_NAMESPACE}")
print(f"Container Image: {CONTAINER_IMAGE}")

In [ ]:
# Initialize Spark session with EMR on EKS and RAPIDS configuration
spark = SparkSession.builder \
    .appName("Fraud Detection Feature Engineering - EMR on EKS") \
    .config("spark.kubernetes.container.image", CONTAINER_IMAGE) \
    .config("spark.kubernetes.namespace", KUBERNETES_NAMESPACE) \
    .config("spark.kubernetes.executor.podNamePrefix", "fraud-detection-fe") \
    .config("spark.kubernetes.driver.podNamePrefix", "fraud-detection-fe-driver") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.executor.memory", "30G") \
    .config("spark.executor.memoryFraction", "0.8") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.instances", "12") \
    .config("spark.executor.resource.gpu.amount", "1") \
    .config("spark.task.resource.gpu.amount", "0.25") \
    .config("spark.shuffle.compress", "true") \
    .config("spark.shuffle.spill.compress", "true") \
    .config("spark.sql.shuffle.partitions", "20000") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.rapids.sql.enabled", "true") \
    .config("spark.plugins", "com.nvidia.spark.SQLPlugin") \
    .config("spark.rapids.memory.pinnedPool.size", "2G") \
    .config("spark.rapids.sql.concurrentGpuTasks", "2") \
    .config("spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict", "false") \
    .getOrCreate()

# Display Spark configuration for verification
network_timeout = spark.conf.get("spark.network.timeout", "Not Set")
max_result_size = spark.conf.get("spark.driver.maxResultSize", "Not Set")
heartbeat_interval = spark.conf.get("spark.executor.heartbeatInterval", "Not Set")
buffer_max = spark.conf.get("spark.kryoserializer.buffer.max")
executor_memory = spark.conf.get("spark.executor.memory")
rapids_enabled = spark.conf.get("spark.rapids.sql.enabled")

print(f"Spark Version: {spark.version}")
print(f"RAPIDS SQL Enabled: {rapids_enabled}")
print(f"Executor Memory: {executor_memory}")
print(f"Kryo Buffer Max: {buffer_max}")
print(f"Network Timeout: {network_timeout}")
print(f"Driver Max Result Size: {max_result_size}")
print(f"Executor Heartbeat Interval: {heartbeat_interval}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Load datasets and infer schema - Same S3 paths as original notebook
customers_path = "s3://nvidia-aws-fraud-detection-demo-training-data/customers_parquet/"
terminals_path = "s3://nvidia-aws-fraud-detection-demo-training-data/terminals_parquet/"
transactions_path = "s3://nvidia-aws-fraud-detection-demo-training-data/transactions_parquet/"

print(f"Loading data from S3...")
print(f"Customers: {customers_path}")
print(f"Terminals: {terminals_path}")
print(f"Transactions: {transactions_path}")

# Load datasets with appropriate partitioning for Kubernetes environment
customers_df = spark.read.parquet(customers_path).repartition(300)
terminals_df = spark.read.parquet(terminals_path)
transactions_df = spark.read.parquet(transactions_path).repartition(1000)

# Show schema of each dataset to understand their structure
print("\nCustomers Schema:")
customers_df.printSchema()

print("\nTerminals Schema:")
terminals_df.printSchema()

print("\nTransactions Schema:")
transactions_df.printSchema()

# Optional: Count rows to verify data loading (uncomment for debugging)
# print(f"Number of rows in customers: {customers_df.count()}")
# print(f"Number of rows in terminals: {terminals_df.count()}")
# print(f"Number of rows in transactions: {transactions_df.count()}")

In [ ]:
# Broadcast smaller tables for efficient joins - Same optimization as original
terminals_df = broadcast(terminals_df)
print("Terminals dataframe broadcasted for efficient joins")

In [ ]:
# Convert the TX_DATETIME column to timestamp and extract date components
# Same logic as original notebook
transactions_df = transactions_df.withColumn(
    "TX_DATETIME",
    F.col("TX_DATETIME").cast("timestamp"))

# Split TX_DATETIME into yyyy, mm, and dd columns
transactions_df = transactions_df.withColumn("yyyy", year(F.col("TX_DATETIME"))) \
                                 .withColumn("mm", month(F.col("TX_DATETIME"))) \
                                 .withColumn("dd", dayofmonth(F.col("TX_DATETIME")))

# Define time windows in seconds for feature extraction - Same as original
time_windows = {
    "15min": 15 * 60,
    "30min": 30 * 60,
    "60min": 60 * 60,
    "1day": 24 * 60 * 60,
    "7day": 7 * 24 * 60 * 60,
    "15day": 15 * 24 * 60 * 60,
    "30day": 30 * 24 * 60 * 60
}

print(f"Time windows defined: {list(time_windows.keys())}")

In [ ]:
# Define a function to add window features efficiently - Same logic as original
def add_window_features(transactions_df, time_windows, entity_id_col, prefix):
    """
    Add time-window based features for customer and terminal analysis.
    This function is identical to the original notebook implementation.
    """
    print(f"Adding window features for {prefix}...")
    
    for window_name, window_duration in time_windows.items():
        window_spec = Window.partitionBy(entity_id_col).orderBy(
            F.col("TX_DATETIME").cast("long")).rangeBetween(
                -window_duration, 0)

        # Number of transactions in the time window
        transactions_df = transactions_df.withColumn(
            f"{prefix}_nb_txns_{window_name}_window",
            F.count("*").over(window_spec))

        # Average transaction amount in the time window
        transactions_df = transactions_df.withColumn(
            f"{prefix}_avg_amt_{window_name}_window",
            F.avg("TX_AMOUNT").over(window_spec))

    return transactions_df

print("Window feature function defined")

In [ ]:
# Add customer-related features - Same as original
print("Processing customer-related window features...")
transactions_df = add_window_features(transactions_df, time_windows,
                                      "CUSTOMER_ID", "customer_id")

# Add terminal-related features - Same as original
print("Processing terminal-related window features...")
transactions_df = add_window_features(transactions_df, time_windows,
                                      "TERMINAL_ID", "terminal_id")

print("Window features processing completed")

In [ ]:
# Ordinal Encoding using StringIndexer - Same logic as original
print("Starting ordinal encoding process...")

# Ordinal Encoding for CUSTOMER_ID
print("Encoding CUSTOMER_ID...")
customer_indexer = StringIndexer(inputCol="CUSTOMER_ID",
                                 outputCol="CUSTOMER_ID_index",
                                 handleInvalid="keep").fit(transactions_df)
transactions_df = customer_indexer.transform(transactions_df)

# Apply the same StringIndexer to customers_df to create the CUSTOMER_ID_index column
customers_df = customer_indexer.transform(customers_df)

# Ordinal encoding for other columns in customers_df
print("Encoding customer attributes...")
columns_to_encode_customers = ['customer_name', 'customer_email', 'phone']
for column in columns_to_encode_customers:
    if column in customers_df.columns:
        print(f"  Encoding {column}...")
        indexer = StringIndexer(inputCol=column,
                                outputCol=f"{column}_index",
                                handleInvalid="keep").fit(customers_df)
        customers_df = indexer.transform(customers_df)

# Ordinal encoding for TERMINAL_ID in transactions_df
print("Encoding TERMINAL_ID...")
terminal_indexer = StringIndexer(inputCol="TERMINAL_ID",
                                 outputCol="TERMINAL_ID_index",
                                 handleInvalid="keep").fit(transactions_df)
transactions_df = terminal_indexer.transform(transactions_df)

# Apply the same StringIndexer to terminals_df to create the TERMINAL_ID_index column
terminals_df = terminal_indexer.transform(terminals_df)

print("Ordinal encoding completed")

In [ ]:
# Handle merchant encoding and additional categorical features - Same as original
print("Processing merchant and additional categorical features...")

# Ordinal encoding for merchant in both transactions_df and terminals_df
if 'merchant' in transactions_df.columns:
    print("Encoding merchant in transactions...")
    merchant_indexer = StringIndexer(inputCol='merchant',
                                     outputCol='merchant_index',
                                     handleInvalid="keep").fit(transactions_df)
    transactions_df = merchant_indexer.transform(transactions_df)

if 'merchant' in terminals_df.columns:
    print("Encoding merchant in terminals...")
    merchant_indexer_terminals = StringIndexer(
        inputCol='merchant', outputCol='merchant_index',
        handleInvalid="keep").fit(terminals_df)
    terminals_df = merchant_indexer_terminals.transform(terminals_df)

# Apply StringIndexer to additional categorical columns in transactions_df
columns_to_encode_transactions = ['merchant']  # Already handled 'merchant'
for column in columns_to_encode_transactions:
    if column in transactions_df.columns:
        print(f"Processing {column} in transactions...")
        indexer = StringIndexer(inputCol=column,
                                outputCol=f"{column}_index",
                                handleInvalid="keep").fit(transactions_df)
        transactions_df = indexer.transform(transactions_df)
        transactions_df = transactions_df.drop(column)

print("Merchant encoding completed")

In [ ]:
# One-hot encoding for TX_FRAUD and additional processing - Same as original
print("Processing fraud labels and final transformations...")

# One-hot encoding for TX_FRAUD
transactions_df = transactions_df.withColumn(
    "TX_FRAUD_0", (F.col("TX_FRAUD") == 0).cast("int"))
transactions_df = transactions_df.withColumn(
    "TX_FRAUD_1", (F.col("TX_FRAUD") == 1).cast("int"))

# Drop TX_FRAUD and TX_DATETIME column after encoding
transactions_df = transactions_df.drop("TX_FRAUD")
transactions_df = transactions_df.drop("TX_DATETIME")

# Apply StringIndexer for billing_city and billing_state in customers_df
print("Encoding billing information...")
billing_city_indexer = StringIndexer(inputCol="billing_city", outputCol="billing_city_index").fit(customers_df)
customers_df = billing_city_indexer.transform(customers_df)

billing_state_indexer = StringIndexer(inputCol="billing_state", outputCol="billing_state_index").fit(customers_df)
customers_df = billing_state_indexer.transform(customers_df)

# Drop the original columns after encoding
customers_df = customers_df.drop("billing_city", "billing_state")

print("Label encoding and transformations completed")

In [ ]:
# Join the enriched transactions data with customer and terminal details - Same as original
print("Performing data joins...")

# Join transactions with customers and terminals using left joins
final_df = transactions_df.join(customers_df,
                                on="CUSTOMER_ID_index",
                                how="left").join(terminals_df,
                                                 on="TERMINAL_ID_index",
                                                 how="left")

print("Data joins completed")

# Optional: Check row count after joins (uncomment for debugging)
# print(f"Total number of rows after joins: {final_df.count()}")

In [ ]:
# Select the final features and customer/terminal details - Same columns as original
print("Selecting final feature columns...")

final_columns = [
    "CUSTOMER_ID_index",
    "customer_name_index",
    "customer_email_index",
    "phone_index",
    "billing_zip",
    "billing_city_index",  # Ordinal encoded billing_city
    "billing_state_index", # Ordinal encoded billing_state
    "x_customer_id",  # Added column
    "y_customer_id",  # Added column
    "TX_AMOUNT",
    "TX_FRAUD_0",  # One-hot encoded column
    "TX_FRAUD_1",  # One-hot encoded column   
    "TERMINAL_ID_index",
    "merchant_index",  # Ensure 'merchant_index' is present
    "yyyy",
    "mm",
    "dd",
    # Customer-related features
    "customer_id_nb_txns_15min_window",
    "customer_id_nb_txns_30min_window",
    "customer_id_nb_txns_60min_window",
    "customer_id_nb_txns_1day_window",
    "customer_id_nb_txns_7day_window",
    "customer_id_nb_txns_15day_window",
    "customer_id_nb_txns_30day_window",
    "customer_id_avg_amt_15min_window",
    "customer_id_avg_amt_30min_window",
    "customer_id_avg_amt_60min_window",
    "customer_id_avg_amt_1day_window",
    "customer_id_avg_amt_7day_window",
    "customer_id_avg_amt_15day_window",
    "customer_id_avg_amt_30day_window",
    # Terminal-related features
    "terminal_id_nb_txns_15min_window",
    "terminal_id_nb_txns_30min_window",
    "terminal_id_nb_txns_60min_window",
    "terminal_id_nb_txns_1day_window",
    "terminal_id_nb_txns_7day_window",
    "terminal_id_nb_txns_15day_window",
    "terminal_id_nb_txns_30day_window",
    "terminal_id_avg_amt_15min_window",
    "terminal_id_avg_amt_30min_window",
    "terminal_id_avg_amt_60min_window",
    "terminal_id_avg_amt_1day_window",
    "terminal_id_avg_amt_7day_window",
    "terminal_id_avg_amt_15day_window",
    "terminal_id_avg_amt_30day_window"
]

# Select final columns and repartition for optimal output
# Adjusted partition count for Kubernetes environment
final_df = final_df.select(final_columns).repartition(5000)

print(f"Final dataset prepared with {len(final_columns)} features")
print("Feature columns:")
for i, col in enumerate(final_columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Save the result to S3 as Parquet - Updated for EMR on EKS environment
print("Configuring output settings...")

# Optimize Spark settings for S3 output in Kubernetes environment
spark.conf.set("spark.sql.files.maxPartitionBytes", "128M")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "500M")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128M")

# Output path - using EMR on EKS specific path
final_output_path = "s3://nvidia-aws-fraud-detection-demo/emr-eks-output/feature-engineering/"

print(f"Writing processed features to: {final_output_path}")
print("This may take several minutes depending on data size and cluster resources...")

# Write to S3 with overwrite mode
final_df.write.mode("overwrite").parquet(final_output_path)

print(f"✅ Data successfully written to {final_output_path}")
print(f"✅ Feature engineering pipeline completed successfully on EMR on EKS")
print(f"✅ Output format: Parquet with {final_df.rdd.getNumPartitions()} partitions")

In [ ]:
# Verification and summary
print("\n=== PIPELINE SUMMARY ===")
print(f"Spark Version: {spark.version}")
print(f"RAPIDS Enabled: {spark.conf.get('spark.rapids.sql.enabled')}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Executor Instances: {spark.conf.get('spark.executor.instances')}")
print(f"GPU Resources per Executor: {spark.conf.get('spark.executor.resource.gpu.amount')}")
print(f"Kubernetes Namespace: {KUBERNETES_NAMESPACE}")
print(f"Container Image: {CONTAINER_IMAGE}")
print(f"Output Location: {final_output_path}")
print(f"Features Generated: {len(final_columns)}")

# Optional: Show sample of final data (uncomment for debugging)
# print("\nSample of processed data:")
# final_df.show(5, truncate=False)

print("\n=== NEXT STEPS ===")
print("1. Verify output data in S3")
print("2. Proceed to ML training notebook (02_fraud_detection_training_ray.ipynb)")
print("3. Monitor Spark job completion in Kubernetes dashboard")
print("4. Check EMR on EKS console for job status and logs")

In [ ]:
# Clean up - Stop the Spark session
print("Stopping Spark session...")
spark.stop()
print("✅ Spark session stopped successfully")
print("✅ EMR on EKS feature engineering job completed")